In [ ]:
from urllib.request import urlopen
import pandas as pd
import json

response = urlopen("https://api.openf1.org/v1/drivers?&session_key=latest")
data = json.loads(response.read().decode('utf-8'))
pd.DataFrame(data)

response = urlopen("https://api.openf1.org/v1/sessions?year=2026")
data = json.loads(response.read().decode('utf-8'))
pd.DataFrame(data)

In [ ]:
def get_session_data(year = 2026):
    response = urlopen(f"https://api.openf1.org/v1/sessions?year={year}")
    data = json.loads(response.read().decode('utf-8'))
    return pd.DataFrame(data)

def get_driver_data(session_key = "latest"):
    response = urlopen(f"https://api.openf1.org/v1/drivers?&session_key={session_key}")
    data = json.loads(response.read().decode('utf-8'))
    return pd.DataFrame(data)

def get_session_results(session_key = "latest"):
    response = urlopen(f"https://api.openf1.org/v1/session_result?session_key={session_key}")
    data = json.loads(response.read().decode('utf-8'))
    return pd.DataFrame(data)

def get_drivers_championship_standings(session_key = "latest"):
    response = urlopen(f"https://api.openf1.org/v1/championship_drivers?session_key={session_key}")
    data = json.loads(response.read().decode('utf-8'))
    return pd.DataFrame(data)

In [ ]:
def get_top_k_position_scores(session_key = "latest", year = 2026, k = 10):
    # Get API data
    before_race_drivers_championship_standings = get_drivers_championship_standings(session_key)[["driver_number", "position_start"]]
    drivers = get_driver_data(session_key)
    session_results = get_session_results(session_key)[["driver_number", "position"]]

    # Merge dataframes to get the final output
    race_output = session_results.merge(
        drivers[["driver_number", "name_acronym"]], 
        left_on="driver_number", 
        right_on="driver_number", 
        how="left"
    ).merge(
        before_race_drivers_championship_standings[["driver_number", "position_start"]],
        on="driver_number",
        how="left"
    )
    return race_output[race_output["position"] <= k].sort_values("position")

In [ ]:
session = get_session_data(year = 2026)

In [ ]:
session

In [ ]:
session = session.sort_values("date_start")
session = session[(session.session_name == "Race") | (session.session_name == "Sprint")]

In [ ]:
round_number_key = session[(session.session_name == "Race")].reset_index(drop=True).reset_index(names="round_number")[["meeting_key","round_number"]]
round_number_key["round_number"] = round_number_key["round_number"] + 1

In [ ]:
round_number_key

In [ ]:
from leaderboard import ResultAggregator

result_aggregator = ResultAggregator(year = 2025, round_number=1)
result_aggregator.aggregate_results()